# Invert for source depth / range

In [ ]:
import os
import pickle
from itertools import combinations

import cmps
import matplotlib.pyplot as plt
import numpy as np
import pygenray as pr
import xarray as xr
from dotenv import find_dotenv, load_dotenv
from geopy.distance import geodesic

from nescafe23_fluctuations import tools
from nescafe23_fluctuations.source_localization import source_localization_iteration
from nescafe23_fluctuations import source_localization as sl
_ = load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

In [ ]:
Tx_idx = 0

## open arrival times

In [ ]:
dss = tools.open_receptions()

In [ ]:
with open(
    f"{os.environ['NESCAFE23_DATADIR']}dataproducts/peak_locations.pkl", "rb"
) as f:
    locations = pickle.load(f)

times = locations["times"]
angles = locations["angles"]
snrs = locations["snrs"]

In [ ]:
ray_ids = list(times.keys())[7:]


difference_keys = list(combinations(ray_ids, 2))

In [ ]:
Delta_Ts_obs = np.ones(len(difference_keys)) * np.nan

for k, diff_key in enumerate(difference_keys):
    Delta_Ts_obs[k] = times[diff_key[0]][Tx_idx] - times[diff_key[1]][Tx_idx]

## Iteration 0

In [ ]:
fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/dataproducts/full_field_new.nc"
full_field = xr.open_dataset(fn)

In [ ]:
front_location = 35e3

envi_field = full_field.assign_coords({
    "range": (full_field.range.values + front_location)
})
c = envi_field["c"]

bathy = xr.DataArray(
    np.linspace(4908, 4908, 100),
    dims=["range"],
    coords={"range": np.linspace(0, 200e3, 100)},
)

envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)

In [ ]:
coords = {
    "SION": [39 + 37.194 / 60, -63 - 39.427 / 60],
    "SIOE": [38 + 45.906 / 60, -62 - 15.626 / 60],
}

source_depth0 = 2300
source_range0 = 0
launch_angles = np.linspace(-35, 35, 6000)
receiver_range0 = geodesic(coords["SIOE"], coords["SION"]).meters
num_range_save = 1000

receiver_depths0 = [1000]

c_source0 = float(
    envi.sound_speed.interp({"range": source_range0, "depth": source_depth0})
)
c_receiever0 = float(
    envi.sound_speed.interp({"range": receiver_range0, "depth": receiver_depths0[0]})
)

Ss0 = 1 / c_source0
Sr0 = 1 / c_receiever0

In [ ]:
rays0 = pr.shoot_rays(
    source_depth0, source_range0, launch_angles, receiver_range0, num_range_save, envi
)
eigs0 = pr.find_eigenrays(
    rays0,
    receiver_depths0,
    source_depth0,
    source_range0,
    receiver_range0,
    num_range_save,
    envi,
)

In [ ]:
times_model0 = {}
angles_model0 = {}
pzs_model0 = {}
pzr_model0 = {}
pxr_model0 = {}
pxs_model0 = {}
px_model0 = {}

for ray_id in ray_ids:
    ray_id_pr, _ = tools.parse_ray_id(ray_id)

    ray_idx = np.where(eigs0.ray_id[0] == ray_id_pr)[0][-1]
    times_model0[ray_id] = eigs0.ts[0][ray_idx, -1]
    angles_model0[ray_id] = eigs0.received_angles[0][ray_idx]

    pzs_model0[ray_id] = eigs0.ps[0][ray_idx, 0]
    pzr_model0[ray_id] = eigs0.ps[0][ray_idx, -1]

    pxr_model0[ray_id] = np.sqrt(Sr0**2 - pzr_model0[ray_id] ** 2)
    pxs_model0[ray_id] = np.sqrt(Ss0**2 - pzs_model0[ray_id] ** 2)

    px_model0[ray_id] = 0.5 * (pxr_model0[ray_id] + pxs_model0[ray_id])

### compute residuals, and Jacobian

In [ ]:
Delta_Ts_sim0 = np.ones(len(difference_keys)) * np.nan

for k, diff_key in enumerate(difference_keys):
    Delta_Ts_sim0[k] = times_model0[diff_key[0]] - times_model0[diff_key[1]]

DDelta_Ts0 = Delta_Ts_obs - Delta_Ts_sim0

In [ ]:
E = np.zeros((len(DDelta_Ts0), 3))

for k in range(len(DDelta_Ts0)):
    E[k, 0] = pzs_model0[difference_keys[k][0]] - pzs_model0[difference_keys[k][1]]
    E[k, 1] = px_model0[difference_keys[k][0]] - px_model0[difference_keys[k][1]]
    E[k, 2] = pzr_model0[difference_keys[k][0]] - pzr_model0[difference_keys[k][1]]

In [ ]:
plt.imshow(E, aspect="auto", interpolation="none")

### compute update

In [ ]:
S = np.diag([100, 4000, 0.5]) ** 2
# 100 [m] for source depth
# 1 [km] for range
# 3 [s] for common time offset

R = np.diag([(10e-3)] * len(DDelta_Ts0)) ** 2
# observation error of 10 [ms]

In [ ]:
y = DDelta_Ts0.copy()
dx_hat = (
    np.linalg.inv(np.linalg.inv(S) + E.T @ np.linalg.inv(R) @ E)
    @ E.T
    @ np.linalg.inv(R)
    @ y
)
P = np.linalg.inv(np.linalg.inv(S) + E.T @ np.linalg.inv(R) @ E)

yerr = E @ P @ E.T

In [ ]:
print(f"Source Depth: {source_depth0 + dx_hat[0]} ± {np.sqrt(P[0, 0])}")
print(f"Range: {receiver_range0 + dx_hat[1]} ± {np.sqrt(P[1, 1])}")
print(f"Time Offset: {dx_hat[2]} ± {np.sqrt(P[2, 2])}")

## Iteration 1

In [ ]:
source_range1 = source_range0
source_depth1 = source_depth0 + dx_hat[0]
receiver_range1 = receiver_range0 + dx_hat[1]
receiver_depths1 = [receiver_depths0[0] + dx_hat[2]]

c_source1 = float(
    envi.sound_speed.interp({"range": source_range1, "depth": source_depth1})
)
c_receiever1 = float(
    envi.sound_speed.interp({"range": receiver_range1, "depth": receiver_depths1[0]})
)

Ss1 = 1 / c_source1
Sr1 = 1 / c_receiever1

In [ ]:
rays1 = pr.shoot_rays(
    source_depth1, source_range1, launch_angles, receiver_range1, num_range_save, envi
)
eigs1 = pr.find_eigenrays(
    rays1,
    receiver_depths1,
    source_depth1,
    source_range1,
    receiver_range1,
    num_range_save,
    envi,
)

In [ ]:
times_model1 = {}
angles_model1 = {}
pzs_model1 = {}
pzr_model1 = {}
pxr_model1 = {}
pxs_model1 = {}
px_model1 = {}

for ray_id in ray_ids:
    ray_id_pr, _ = tools.parse_ray_id(ray_id)

    ray_idx = np.where(eigs1.ray_id[0] == ray_id_pr)[0][-1]
    times_model1[ray_id] = eigs1.ts[0][ray_idx, -1]
    angles_model1[ray_id] = eigs1.received_angles[0][ray_idx]

    pzs_model1[ray_id] = eigs1.ps[0][ray_idx, 0]
    pzr_model1[ray_id] = eigs1.ps[0][ray_idx, -1]

    pxr_model1[ray_id] = np.sqrt(Sr1**2 - pzr_model1[ray_id] ** 2)
    pxs_model1[ray_id] = np.sqrt(Ss1**2 - pzs_model1[ray_id] ** 2)

    px_model1[ray_id] = 0.5 * (pxr_model1[ray_id] + pxs_model1[ray_id])

### compute residuals, and Jacobian

In [ ]:
Delta_Ts_sim1 = np.ones(len(difference_keys)) * np.nan

for k, diff_key in enumerate(difference_keys):
    Delta_Ts_sim1[k] = times_model1[diff_key[0]] - times_model1[diff_key[1]]

DDelta_Ts1 = Delta_Ts_obs - Delta_Ts_sim1

In [ ]:
E1 = np.zeros((len(DDelta_Ts1), 3))

for k in range(len(DDelta_Ts1)):
    E1[k, 0] = pzs_model1[difference_keys[k][0]] - pzs_model1[difference_keys[k][1]]
    E1[k, 1] = px_model1[difference_keys[k][0]] - px_model1[difference_keys[k][1]]
    E1[k, 2] = pzr_model1[difference_keys[k][0]] - pzr_model1[difference_keys[k][1]]

In [ ]:
plt.imshow(E1, aspect="auto", interpolation="none")

### compute update

In [ ]:
y1 = DDelta_Ts0.copy()
dx_hat1 = (
    np.linalg.inv(np.linalg.inv(S) + E1.T @ np.linalg.inv(R) @ E1)
    @ E1.T
    @ np.linalg.inv(R)
    @ y1
)
P1 = np.linalg.inv(np.linalg.inv(S) + E1.T @ np.linalg.inv(R) @ E1)

yerr1 = E1 @ P @ E1.T

In [ ]:
print(f"Source Depth: {source_depth1 + dx_hat1[0]} ± {np.sqrt(P1[0, 0])}")
print(f"Range: {receiver_range1 + dx_hat1[1]} ± {np.sqrt(P1[1, 1])}")
print(f"Time Offset: {dx_hat1[2]} ± {np.sqrt(P1[2, 2])}")

## Iteration 2

In [ ]:
source_range2 = source_range1
source_depth2 = source_depth1 + dx_hat1[0]
receiver_range2 = receiver_range1 + dx_hat1[1]
receiver_depths2 = [receiver_depths1[0] + dx_hat1[2]]

c_source2 = float(
    envi.sound_speed.interp({"range": source_range2, "depth": source_depth2})
)
c_receiever2 = float(
    envi.sound_speed.interp({"range": receiver_range2, "depth": receiver_depths2[0]})
)

Ss2 = 1 / c_source2
Sr2 = 1 / c_receiever2

In [ ]:
rays2 = pr.shoot_rays(
    source_depth2, source_range2, launch_angles, receiver_range2, num_range_save, envi
)
eigs2 = pr.find_eigenrays(
    rays2,
    receiver_depths2,
    source_depth2,
    source_range2,
    receiver_range2,
    num_range_save,
    envi,
)

In [ ]:
times_model2 = {}
angles_model2 = {}
pzs_model2 = {}
pzr_model2 = {}
pxr_model2 = {}
pxs_model2 = {}
px_model2 = {}

for ray_id in ray_ids:
    ray_id_pr, _ = tools.parse_ray_id(ray_id)

    ray_idx = np.where(eigs2.ray_id[0] == ray_id_pr)[0][-1]
    times_model2[ray_id] = eigs2.ts[0][ray_idx, -1]
    angles_model2[ray_id] = eigs2.received_angles[0][ray_idx]

    pzs_model2[ray_id] = eigs2.ps[0][ray_idx, 0]
    pzr_model2[ray_id] = eigs2.ps[0][ray_idx, -1]

    pxr_model2[ray_id] = np.sqrt(Sr2**2 - pzr_model2[ray_id] ** 2)
    pxs_model2[ray_id] = np.sqrt(Ss2**2 - pzs_model2[ray_id] ** 2)

    px_model2[ray_id] = 0.5 * (pxr_model2[ray_id] + pxs_model2[ray_id])

### compute residuals, and Jacobian

In [ ]:
Delta_Ts_sim2 = np.ones(len(difference_keys)) * np.nan

for k, diff_key in enumerate(difference_keys):
    Delta_Ts_sim2[k] = times_model2[diff_key[0]] - times_model2[diff_key[1]]

DDelta_Ts2 = Delta_Ts_obs - Delta_Ts_sim2

In [ ]:
plt.plot(DDelta_Ts0)
plt.plot(DDelta_Ts1)
plt.plot(DDelta_Ts2)

In [ ]:
E1 = np.zeros((len(DDelta_Ts1), 3))

for k in range(len(DDelta_Ts1)):
    E1[k, 0] = pzs_model1[difference_keys[k][0]] - pzs_model1[difference_keys[k][1]]
    E1[k, 1] = px_model1[difference_keys[k][0]] - px_model1[difference_keys[k][1]]
    E1[k, 2] = pzr_model1[difference_keys[k][0]] - pzr_model1[difference_keys[k][1]]

In [ ]:
plt.imshow(E1, aspect="auto", interpolation="none")

### compute update

In [ ]:
y1 = DDelta_Ts0.copy()
dx_hat1 = (
    np.linalg.inv(np.linalg.inv(S) + E1.T @ np.linalg.inv(R) @ E1)
    @ E1.T
    @ np.linalg.inv(R)
    @ y1
)
P1 = np.linalg.inv(np.linalg.inv(S) + E1.T @ np.linalg.inv(R) @ E1)

yerr1 = E1 @ P @ E1.T

In [ ]:
print(f"Source Depth: {source_depth1 + dx_hat1[0]} ± {np.sqrt(P1[0, 0])}")
print(f"Range: {receiver_range1 + dx_hat1[1]} ± {np.sqrt(P1[1, 1])}")
print(f"Time Offset: {dx_hat1[2]} ± {np.sqrt(P1[2, 2])}")

# Build Source Localization Function

In [ ]:
## Open Observed Arrival Times
Tx_idx = 0

with open(
    f"{os.environ['NESCAFE23_DATADIR']}dataproducts/peak_locations.pkl", "rb"
) as f:
    locations = pickle.load(f)

times = locations["times"]
angles = locations["angles"]
snrs = locations["snrs"]

ray_ids = list(times.keys())[7:]
difference_keys = list(combinations(ray_ids, 2))

Delta_Ts_obs = np.ones(len(difference_keys)) * np.nan

for k, diff_key in enumerate(difference_keys):
    Delta_Ts_obs[k] = times[diff_key[0]][Tx_idx] - times[diff_key[1]][Tx_idx]

In [ ]:
## Create model environment object
fn = f"{os.environ['NESCAFE23_DATADIR']}environment_data/dataproducts/full_field_nlm.nc"
full_field = xr.open_dataset(fn)

front_location = 35e3

envi_field = full_field.assign_coords({
    "range": (full_field.range.values + front_location)
})
c = envi_field["c"]

bathy = xr.DataArray(
    np.linspace(4908, 4908, 100),
    dims=["range"],
    coords={"range": np.linspace(0, 200e3, 100)},
)

envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)

In [ ]:
## Initial Guess and Priors

source_depth0 = 1500  # [m]
reciever_range0 = 153e3  # [m]
reciever_depth0 = 1000  # [m]

xs = [np.array([source_depth0, reciever_range0, reciever_depth0])]

S = np.diag([100, 4000, 0.5]) ** 2
# 100 [m] for source depth
# 1 [km] for range
# 3 [s] for common time offset

R = 2 * np.diag([(10e-3)] * len(difference_keys)) ** 2
# observation error of 10 [ms]

## iteration loop

In [ ]:
source_depth0 = 1800  # [m]
reciever_range0 = 153e3  # [m]
reciever_depth0 = 1000  # [m]

x0 = np.array([source_depth0, reciever_range0, reciever_depth0])

Delta_Ts_sim = []
E = []
y = []
dx = []
P = []
rays = []
eigs = []
x = [x0]

for k in range(5):
    print(
        f"Iteration {k}: Source Depth = {x[k][0]}, Range = {x[k][1]}, Reciever Depth = {x[k][2]}"
    )
    Delta_Ts_simi, Ei, yi, dxi, Pi, raysi, eigsi = source_localization_iteration(
        x[k], S, R, envi, ray_ids, difference_keys, Delta_Ts_obs
    )

    Delta_Ts_sim.append(Delta_Ts_simi)
    E.append(Ei)
    y.append(yi)
    dx.append(dxi)
    P.append(Pi)
    rays.append(raysi)
    eigs.append(eigsi)

    # compute update for next iteration
    x.append(x[k] + dx[k])

In [ ]:
plt.errorbar(
    np.arange(len(difference_keys)), y[-1], np.sqrt(np.diag(R)), linestyle="none"
)

In [ ]:
for k,y_single in enumerate(y[:3]):
    plt.plot(y_single*1000, label=f'Iteration {k+1}')

plt.legend()
plt.grid()
plt.ylabel('Residuals [ms]')
plt.xlabel('Time Difference Index')

plt.ylim([-250,250])

In [ ]:
source_depth0 = 1800  # [m]
reciever_range0 = 153e3  # [m]
reciever_depth0 = 1000  # [m]

x0 = np.array([source_depth0, reciever_range0, reciever_depth0])
Delta_Ts_sim0, E0, y0, dx0, P0, rays0, eigs0 = tools.source_localization_iteration(
    x0, S, R, envi, ray_ids, difference_keys, Delta_Ts_obs
)

In [ ]:
x1 = xs0 + dx0
Delta_Ts_sim1, E1, y1, dx1, P1, rays1, eigs1 = tools.source_localization_iteration(
    x1, S, R, envi, ray_ids, difference_keys, Delta_Ts_obs
)

In [ ]:
x2 = x1 + dx1
Delta_Ts_sim2, E2, y2, dx2, P2, rays2, eigs2 = tools.source_localization_iteration(
    x2, S, R, envi, ray_ids, difference_keys, Delta_Ts_obs
)

In [ ]:
x3 = x2 + dx2
Delta_Ts_sim1, E3, y3, dx3, P3, rays3, eigs3 = tools.source_localization_iteration(
    x3, S, R, envi, ray_ids, difference_keys, Delta_Ts_obs
)

In [ ]:
x3

In [ ]:
plt.plot(y0)
plt.plot(y1)
plt.plot(y2)
plt.plot(y3)

In [ ]:
plt.plot(y0)
plt.plot(y1)
plt.plot(y2)
plt.plot(y3)

In [ ]:
fig = plt.figure(figsize=(16, 3))
(20 * np.log10(np.abs(dss[Tx_idx][32:, :][:, ::10]))).swap_dims({
    "hydrophone": "actual_depth"
}).plot(vmin=100, cmap=cmps.plasma_coolwarm())

# plt.plot(rays0.ts[:,-1]-0.25, -rays0.zs[:,-1], lw=1, c='r')
plt.plot(rays3.ts[:, -1] - 0.25, -rays3.zs[:, -1], lw=1, c="k", alpha=0.3)
plt.scatter(rays3.ts[:, -1] - 0.25, -rays3.zs[:, -1], lw=0, s=3, c="k")


plt.ylim([1250, 900])
plt.xlim([100, 115])

In [ ]:
fig = plt.figure(figsize=(16, 3))
(20 * np.log10(np.abs(dss[Tx_idx][32:, :]))).swap_dims({
    "hydrophone": "actual_depth"
}).plot(vmin=100, cmap=cmps.plasma_coolwarm())

# plt.plot(rays0.ts[:,-1]-0.25, -rays0.zs[:,-1], lw=1, c='r')
plt.plot(rays3.ts[:, -1] - 0.25, -rays3.zs[:, -1], lw=1, c="k", alpha=0.3)
plt.scatter(rays3.ts[:, -1] - 0.25, -rays3.zs[:, -1], lw=0, s=3, c="k")


plt.ylim([1250, 900])
plt.xlim([100, 102])

In [ ]:
eigs3.plot_ducted()

# Build source localization section of CLI

In [ ]:
from nescafe23_fluctuations import source_localization as sl

In [ ]:
E_all, y_all, x_all, P_all, rays_all, eigs_all = sl.localize_n32_source()

In [ ]:
# save to pickle file for debug later

In [ ]:
with open('source_localization.pkl', 'wb') as f:
    pickle.dump({'E_all':E_all, 'y_all':y_all, 'x_all':x_all, 'P_all':P_all, 'rays_all':rays_all, 'eigs_all':eigs_all}, f)

# Source Localization through FCTD

In [ ]:
## Open Observed Arrival Times
Tx_idx = 0

with open(
    f"{os.environ['NESCAFE23_DATADIR']}dataproducts/peak_locations.pkl", "rb"
) as f:
    locations = pickle.load(f)

times = locations["times"]
angles = locations["angles"]
snrs = locations["snrs"]


ray_ids = list(times.keys())[7:-6]
difference_keys = list(combinations(ray_ids, 2))

Delta_Ts_obs = np.ones(len(difference_keys)) * np.nan

for k, diff_key in enumerate(difference_keys):
    Delta_Ts_obs[k] = times[diff_key[0]][Tx_idx] - times[diff_key[1]][Tx_idx]

In [ ]:
## Create model environment object
full_field = tools.compute_filtered_fctd_grid(z_smooth=300, r_smooth=5000)

front_location = 35e3

envi_field = full_field.assign_coords({
    "range": (full_field.range.values + front_location)
})
c = envi_field["c"]

bathy = xr.DataArray(
    np.linspace(5000, 5000, 100),
    dims=["range"],
    coords={"range": np.linspace(0, 200e3, 100)},
)

envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)

In [ ]:
## Initial Guess and Priors

source_depth0 = 1000  # [m]
reciever_range0 = 153e3  # [m]
reciever_depth0 = 1000  # [m]

xs = [np.array([source_depth0, reciever_range0, reciever_depth0])]

S = np.diag([500, 4000, 500]) ** 2
# 100 [m] for source depth
# 1 [km] for range
# 3 [s] for common time offset

R = 2 * np.diag([(20e-3)] * len(difference_keys)) ** 2
# observation error of 10 [ms]

## iteration loop

In [ ]:
x0 = np.array([source_depth0, reciever_range0, reciever_depth0])

Delta_Ts_sim = []
E = []
y = []
dx = []
P = []
rays = []
eigs = []
x = [x0]

for k in range(5):
    print(
        f"Iteration {k}: Source Depth = {x[k][0]}, Range = {x[k][1]}, Reciever Depth = {x[k][2]}"
    )
    Delta_Ts_simi, Ei, yi, dxi, Pi, raysi, eigsi = source_localization_iteration(
        x[k], S, R, envi, ray_ids, Delta_Ts_obs
    )

    Delta_Ts_sim.append(Delta_Ts_simi)
    E.append(Ei)
    y.append(yi)
    dx.append(dxi)
    P.append(Pi)
    rays.append(raysi)
    eigs.append(eigsi)

    # compute update for next iteration
    x.append(x[k] + dx[k])

In [ ]:
P[-1]

In [ ]:
_ = sl.plot_marginal_gaussian(x[-1], P[-1], (0,1))

In [ ]:
source_depths = []
ranges = []
r_depths = []

for k in range(5):
    source_depths.append(x[k][0])
    ranges.append(x[k][1])
    r_depths.append(x[k][2])

fig = plt.figure(figsize=(10,4))

plt.subplot(1,3,1)
plt.plot(source_depths, marker='o')

plt.subplot(1,3,2)
plt.plot(ranges, marker='o')

plt.subplot(1,3,3)
plt.plot(r_depths, marker='o')

plt.tight_layout()

In [ ]:
dss[0].actual_depth[32:].mean()

In [ ]:
for k,y_single in enumerate(y[:3]):
    plt.plot(y_single*1000, label=f'Iteration {k+1}')

plt.legend()
plt.grid()
plt.ylabel('Residuals [ms]')
plt.xlabel('Time Difference Index')

plt.ylim([-100,100])

In [ ]:
np.std(y[-1])

In [ ]:
fig = plt.figure(figsize=(16, 3))
(20 * np.log10(np.abs(dss[Tx_idx][32:, :][:, ::10]))).swap_dims({
    "hydrophone": "actual_depth"
}).plot(vmin=100, cmap=cmps.plasma_coolwarm())

# plt.plot(rays0.ts[:,-1]-0.25, -rays0.zs[:,-1], lw=1, c='r')
#plt.plot(rays[3].ts[:, -1] + 0.34, -rays[3].zs[:, -1], lw=1, c="k", alpha=0.3)
plt.scatter(rays[3].ts[:, -1] + 0.34, -rays[3].zs[:, -1], lw=0, s=5, c="k")


#plt.ylim([1250, 900])
#plt.xlim([100, 102])